# Document Question Answering System (RAG)

A Retrieval-Augmented Generation (RAG) pipeline that answers questions about
your own documents (PDFs or text files) instead of relying only on a language
model's built-in knowledge.

**Pipeline:** Load documents -> chunk text -> embed chunks -> store in a vector
database -> embed the user's question -> retrieve the most relevant chunks ->
ask Claude to answer using only that retrieved context.

**Stack used in this notebook:**

| Stage | Tool |
|---|---|
| Embedding model | `sentence-transformers` (`all-MiniLM-L6-v2`, runs locally, free) |
| Vector database | `ChromaDB` (local, file-based, no server needed) |
| Language model | Claude API (`claude-sonnet-4-6`) |

### Before you run this
1. Install dependencies (see the next cell).
2. Set your Anthropic API key as an environment variable called `ANTHROPIC_API_KEY`,
   or paste it into the `client = anthropic.Anthropic(...)` cell below.
3. Put your own PDF or `.txt` files in the `sample_docs/` folder (a sample file
   about RAG itself is already included so you can test the pipeline immediately).


## 1. Install & import dependencies

In [ ]:
# Run this once. If you're in Jupyter, the leading "!" runs it as a shell command.
!pip install sentence-transformers chromadb anthropic pypdf -q


In [ ]:
import os
import glob
import uuid

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
import anthropic

print("All libraries imported successfully.")


## 2. Configuration

Set the folder where your documents live, the chunk size, and your API key here.


In [ ]:
# --- Configuration ---
DOCS_FOLDER = "sample_docs"          # folder containing your .pdf and .txt files
CHUNK_SIZE = 800                     # characters per chunk
CHUNK_OVERLAP = 150                  # characters of overlap between consecutive chunks
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
CLAUDE_MODEL = "claude-sonnet-4-6"
TOP_K = 4                            # how many chunks to retrieve per question

# Set your Anthropic API key as an environment variable before running this notebook:
#   export ANTHROPIC_API_KEY="sk-ant-..."
# Or, uncomment the line below and paste your key directly (not recommended for shared notebooks).
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment
print("Config set. Docs folder:", DOCS_FOLDER)


## 3. Document ingestion

Load every `.pdf` and `.txt` file from `DOCS_FOLDER` and convert it into plain text.


In [ ]:
def load_text_file(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def load_pdf_file(path):
    reader = PdfReader(path)
    text_parts = []
    for page in reader.pages:
        text_parts.append(page.extract_text() or "")
    return "\n".join(text_parts)


def load_documents(folder):
    '''Returns a list of dicts: {"filename": ..., "text": ...}'''
    documents = []
    paths = glob.glob(os.path.join(folder, "*.txt")) + glob.glob(os.path.join(folder, "*.pdf"))

    if not paths:
        print(f"No .txt or .pdf files found in '{folder}'. Add some files and re-run this cell.")
        return documents

    for path in paths:
        filename = os.path.basename(path)
        if path.lower().endswith(".pdf"):
            text = load_pdf_file(path)
        else:
            text = load_text_file(path)

        if text.strip():
            documents.append({"filename": filename, "text": text})
            print(f"Loaded {filename} ({len(text)} characters)")
        else:
            print(f"Skipped {filename}: no extractable text")

    return documents


documents = load_documents(DOCS_FOLDER)
print(f"\nTotal documents loaded: {len(documents)}")


## 4. Text chunking

Long documents are split into smaller overlapping chunks. Smaller chunks make
retrieval more precise; the overlap prevents ideas from being cut off awkwardly
at a chunk boundary.


In [ ]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    '''Splits text into overlapping chunks of roughly `chunk_size` characters.'''
    text = " ".join(text.split())  # normalize whitespace
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        if end >= len(text):
            break
        start = end - overlap  # step back to create overlap
    return chunks


def chunk_documents(documents):
    '''Returns a list of dicts: {"id": ..., "text": ..., "source": ...}'''
    all_chunks = []
    for doc in documents:
        doc_chunks = chunk_text(doc["text"])
        for i, chunk in enumerate(doc_chunks):
            all_chunks.append({
                "id": str(uuid.uuid4()),
                "text": chunk,
                "source": doc["filename"],
                "chunk_index": i,
            })
    return all_chunks


chunks = chunk_documents(documents)
print(f"Total chunks created: {len(chunks)}")
if chunks:
    print("\nExample chunk:")
    print("  Source:", chunks[0]["source"])
    print("  Text:", chunks[0]["text"][:200] + "...")


## 5. Embedding creation

Each chunk is converted into a vector (a list of numbers) that captures its
semantic meaning. Chunks with similar meaning end up with similar vectors,
which is what makes semantic search possible.

The first time you run this cell, `sentence-transformers` will download the
model (~80MB) from Hugging Face -- this requires an internet connection.


In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"Loaded embedding model '{EMBEDDING_MODEL_NAME}'")
print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")


## 6. Vector database (ChromaDB)

We create a local, persistent ChromaDB collection and store every chunk's
embedding, text, and metadata in it. ChromaDB handles the similarity search
for us once everything is loaded.


In [ ]:
# Persistent client: data is saved to disk in ./chroma_db so you don't have to
# re-embed everything every time you restart the notebook.
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Recreate the collection fresh each time this cell runs, so re-running ingestion
# doesn't create duplicate entries.
collection_name = "documents"
try:
    chroma_client.delete_collection(collection_name)
except Exception:
    pass
collection = chroma_client.create_collection(name=collection_name)

if chunks:
    texts = [c["text"] for c in chunks]
    ids = [c["id"] for c in chunks]
    metadatas = [{"source": c["source"], "chunk_index": c["chunk_index"]} for c in chunks]

    print("Generating embeddings for all chunks...")
    embeddings = embedding_model.encode(texts, show_progress_bar=True, normalize_embeddings=True).tolist()

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=texts,
        metadatas=metadatas,
    )
    print(f"\nStored {collection.count()} chunks in the vector database.")
else:
    print("No chunks to store. Add documents to the docs folder and re-run from Step 3.")


## 7. Query processing & context retrieval

When the user asks a question, we embed the question with the *same* embedding
model, then ask ChromaDB for the `TOP_K` most similar chunks.


In [ ]:
def retrieve_context(question, top_k=TOP_K):
    '''Embeds the question and retrieves the most relevant chunks from ChromaDB.'''
    question_embedding = embedding_model.encode([question], normalize_embeddings=True).tolist()

    results = collection.query(
        query_embeddings=question_embedding,
        n_results=top_k,
    )

    retrieved = []
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    distances = results["distances"][0]

    for text, meta, dist in zip(docs, metas, distances):
        retrieved.append({
            "text": text,
            "source": meta["source"],
            "chunk_index": meta["chunk_index"],
            "distance": dist,
        })
    return retrieved


# Quick test of retrieval on its own (no LLM call yet)
test_question = "What is RAG and why is it useful?"
test_results = retrieve_context(test_question)

print(f"Question: {test_question}\n")
for i, r in enumerate(test_results, 1):
    print(f"--- Retrieved chunk {i} (source: {r['source']}, distance: {r['distance']:.4f}) ---")
    print(r["text"][:250] + "...\n")


## 8. Answer generation

The retrieved chunks are inserted into a prompt as context, and Claude
generates the final answer. The prompt explicitly instructs Claude to answer
*only* from the provided context and to say so if the answer isn't present,
which keeps responses grounded and reduces hallucination.


In [ ]:
def build_prompt(question, retrieved_chunks):
    context_block = "\n\n".join(
        f"[Source: {c['source']}, chunk {c['chunk_index']}]\n{c['text']}"
        for c in retrieved_chunks
    )

    prompt = f'''You are a helpful assistant that answers questions using ONLY the
context provided below, which was retrieved from the user's own documents.

- If the answer is fully contained in the context, answer it clearly and concisely.
- If the context only partially answers the question, answer what you can and say
  what information is missing.
- If the context does not contain the answer at all, say so plainly instead of
  guessing or using outside knowledge.
- When helpful, mention which source the information came from.

Context:
{context_block}

Question: {question}

Answer:'''
    return prompt


def answer_question(question, top_k=TOP_K, verbose=True):
    retrieved_chunks = retrieve_context(question, top_k=top_k)

    if not retrieved_chunks:
        return "No documents have been indexed yet. Add files to the docs folder and run Steps 3-6."

    prompt = build_prompt(question, retrieved_chunks)

    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}],
    )

    answer = response.content[0].text

    if verbose:
        sources = set(c["source"] for c in retrieved_chunks)
        print(f"Question: {question}\n")
        print(f"Retrieved {len(retrieved_chunks)} chunks from: {sources}\n")
        print(f"Answer:\n{answer}")

    return answer


# Try it out
_ = answer_question("What is the main idea of the document?")


## 9. Ask your own questions

Use the cell below as an interactive Q&A loop. Edit `my_question` and re-run,
or wrap this in a `while True` loop with `input()` if you want a live chat
experience.


In [ ]:
my_question = "What are the key components of a RAG system?"
_ = answer_question(my_question)


In [ ]:
# Optional: simple interactive loop. Run this cell, type questions, type
# "exit" or "quit" to stop. (Works in classic Jupyter / JupyterLab; in some
# notebook environments you may need to use the cell above instead.)

# while True:
#     q = input("Ask a question (or 'exit'): ")
#     if q.strip().lower() in {"exit", "quit"}:
#         break
#     answer_question(q)
#     print("\n" + "="*80 + "\n")


## 10. Inspect retrieval quality (optional debugging)

If an answer seems wrong or incomplete, the most common cause is that the
*retrieval* step pulled the wrong chunks -- not that the language model is
"bad." This cell lets you inspect exactly what was retrieved for any question,
which is the most useful debugging step in any RAG system.


In [ ]:
def inspect_retrieval(question, top_k=TOP_K):
    results = retrieve_context(question, top_k=top_k)
    print(f"Question: {question}\n")
    for i, r in enumerate(results, 1):
        print(f"#{i} | source: {r['source']} | chunk {r['chunk_index']} | distance: {r['distance']:.4f}")
        print(r["text"])
        print("-" * 80)


inspect_retrieval("What are the limitations of RAG?")


## Next steps / experiments

Ideas from the project brief you can try here:

- **Chunking strategies**: try sentence- or paragraph-based chunking instead of
  fixed character windows (e.g. split on blank lines first, then merge small paragraphs).
- **Different embedding models**: swap `EMBEDDING_MODEL_NAME` for `all-mpnet-base-v2`
  (higher quality, slower) or a hosted model like Voyage AI's `voyage-3`.
- **Hybrid search**: combine ChromaDB's vector search with simple keyword
  matching (e.g. `rank_bm25`) and merge the two result sets.
- **Re-ranking**: after retrieving the top 10-20 chunks by vector similarity,
  use a cross-encoder model to re-score and keep only the best few.
- **Multiple documents at once**: drop several PDFs into `sample_docs/` and
  ask questions that require combining information across files.
- **Citations**: modify `build_prompt` to require Claude to cite the exact
  source filename for every claim in its answer.
